# Introduction à JAX

JAX fournit des tableaux numériques proches de ceux de NumPy, auxquels il ajoute notamment la différentiation automatique et des transformations de fonctions. Nous l'abordons ici comme un outil de calcul pour les machines mathématiques.

À la fin de ce notebook, vous devriez savoir :

- créer et manipuler des tableaux JAX ;
- distinguer produit composante par composante et produit matriciel ;
- modifier fonctionnellement un tableau immuable ;
- calculer un gradient, une hessienne ou une matrice jacobienne ;
- employer `vmap`, `jit` et les clés aléatoires ;
- effectuer quelques pas de gradient sur une machine affine.

## Exercices

1. [Produits euclidien et de Frobenius](#Exercice-1)
2. [Mises à jour fonctionnelles](#Exercice-2)
3. [Dérivée directionnelle et hessienne](#Exercice-3)
4. [Identification d'une machine affine](#Exercice-4)

## 1. Tableaux et calcul matriciel

Par convention, nous importons `jax.numpy` sous le nom `jnp`. La plupart des opérations élémentaires ont la même syntaxe qu'avec NumPy. Nous activons ici les nombres flottants sur 64 bits, utiles pour le calcul scientifique. Cette configuration doit être faite au début du notebook.

In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

print(f"JAX {jax.__version__}")

JAX 0.7.0


Un tableau (`'array'`) possède notamment une forme (`shape`) et un type numérique (`dtype`). L'opérateur `*` agit composante par composante ; l'opérateur `@` représente la composition matricielle.

In [2]:
x = jnp.array([1.0, 2.0, -1.0])
z = jnp.array([2.0, -1.0, 3.0])
A = jnp.array([[1.0, 2.0, 0.0], [0.0, -1.0, 3.0]])

print(f"{x.shape=} {x.dtype=}")
print(f"{A.shape=} {A.dtype=}")
print(f"x * z = {x * z}")             # produit de Hadamard
print(f"A @ x = {A @ x}")             # produit matrice-vecteur
print(f"A.T.shape = {A.T.shape}")      # transposée

x.shape=(3,) x.dtype=dtype('float64')
A.shape=(2, 3) A.dtype=dtype('float64')
x * z = [ 2. -2. -3.]
A @ x = [ 5. -5.]
A.T.shape = (3, 2)


### Exercice 1

**Produits euclidien et de Frobenius.**

Choisissez $y\in\mathbb R^2$ et vérifiez numériquement l'identité

$$
\langle y,Ax\rangle=\langle yx^\top,A\rangle_F.
$$

On pourra calculer le membre de droite avec `jnp.sum((y[:, None] * x[None, :]) * A)`. Expliquez les formes des tableaux intermédiaires.

In [ ]:
# À compléter.

## 2. Immutabilité des tableaux

Contrairement aux tableaux NumPy, les tableaux JAX sont immuables. Une instruction comme `B[0, 1] = 2` provoque donc une erreur. La notation `.at` construit un nouveau tableau et laisse l'ancien inchangé.

In [3]:
B = jnp.zeros((3, 3))
B1 = B.at[0, 1].set(2.0)
B2 = B1.at[:, 0].add(jnp.array([1.0, 2.0, 3.0]))

print("B =\n", B)
print("B1 =\n", B1)
print("B2 =\n", B2)

B =
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
B1 =
 [[0. 2. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
B2 =
 [[1. 2. 0.]
 [2. 0. 0.]
 [3. 0. 0.]]


### Exercice 2

**Mises à jour fonctionnelles.**

À partir de la matrice nulle de taille $4\times4$, construisez sans boucle :

1. la matrice dont la diagonale vaut $(1,2,3,4)$ ;
2. puis une nouvelle matrice obtenue en ajoutant $-1$ à sa dernière colonne.

Vérifiez après chaque opération que la matrice précédente n'a pas changé.

In [ ]:
# À compléter.

## 3. Différentiation automatique

La transformation `jax.grad` s'applique à une fonction scalaire et renvoie une fonction calculant son gradient. L'argument `argnums` indique par rapport à quel argument on différentie. Avec `jax.hessian`, on obtient la hessienne.

Les transformations de JAX sont particulièrement simples pour des fonctions pures (`'pure functions'`) : leur résultat ne dépend que de leurs arguments et elles ne modifient pas d'état extérieur.

In [4]:
Q = jnp.array([[3.0, 1.0], [1.0, 2.0]])
b = jnp.array([1.0, -1.0])

def energie(x, Q, b):
    return 0.5 * x @ Q @ x - b @ x

gradient_energie = jax.grad(energie, argnums=0)
hessienne_energie = jax.hessian(energie, argnums=0)

x0 = jnp.array([2.0, -1.0])
g0 = gradient_energie(x0, Q, b)
H0 = hessienne_energie(x0, Q, b)

print(f"gradient = {g0}")
print(f"hessienne =\n{H0}")
assert jnp.allclose(g0, Q @ x0 - b)
assert jnp.allclose(H0, Q)

gradient = [4. 1.]
hessienne =
[[3. 1.]
 [1. 2.]]


### Exercice 3

**Dérivée directionnelle et hessienne.**

Pour $p=(1,-2)^\top$, comparez $\nabla E(x_0)^\top p$ avec

$$
\frac{E(x_0+sp)-E(x_0)}{s}
$$

pour $s=10^{-1},10^{-2},\ldots,10^{-8}$. Interprétez la diminution initiale de l'erreur, puis sa possible remontée. Vérifiez également la hessienne à partir de la fonction `gradient_energie` avec `jax.jacfwd`.

In [ ]:
# À compléter.

### Fonctions à valeurs vectorielles

Pour une fonction à valeurs vectorielles, on calcule une matrice jacobienne plutôt qu'un gradient. JAX propose notamment les transformations directe `jax.jacfwd` et adjointe `jax.jacrev`. Elles donnent ici le même résultat, mais leur coût peut différer selon les dimensions de départ et d'arrivée.

In [5]:
def F(x):
    return jnp.array([x[0] * x[1], jnp.sin(x[0]) + x[1] ** 2])

DF_directe = jax.jacfwd(F)
DF_adjointe = jax.jacrev(F)

print(DF_directe(x0))
assert jnp.allclose(DF_directe(x0), DF_adjointe(x0))

[[-1.          2.        ]
 [-0.41614684 -2.        ]]


## 4. Paramètres structurés et `value_and_grad`

Les paramètres d'une machine sont rarement réunis dans un seul vecteur. JAX sait manipuler des listes, des tuples et des dictionnaires imbriqués de tableaux, appelés arborescences Python (`'PyTrees'`). Le gradient possède alors la même structure que les paramètres.

La transformation `jax.value_and_grad` calcule simultanément la valeur d'une fonction et son gradient.

In [6]:
def machine_affine(parametres, x):
    return parametres["a"] * x + parametres["b"]

def cout_quadratique(parametres, x, z):
    residu = machine_affine(parametres, x) - z
    return 0.5 * jnp.mean(residu**2)

x_observe = jnp.linspace(-1.0, 1.0, 9)
z_observe = 2.0 * x_observe - 1.0
parametres = {"a": jnp.array(0.0), "b": jnp.array(0.0)}

valeur, gradient = jax.value_and_grad(cout_quadratique)(
    parametres, x_observe, z_observe
)
print(f"coût = {valeur}")
print(f"gradient = {gradient}")

coût = 1.3333333333333333
gradient = {'a': Array(-0.83333333, dtype=float64, weak_type=True), 'b': Array(1., dtype=float64, weak_type=True)}


## 5. Vectorisation avec `vmap`

La transformation `vmap` applique automatiquement une fonction à un lot (`'batch'`) d'arguments. Dans `in_axes=(None, 0)`, `None` signifie que les paramètres restent fixes et `0` que l'on parcourt le premier axe de `x`.

In [7]:
def machine_scalaire(parametres, x):
    return jnp.tanh(parametres["a"] * x + parametres["b"])

machine_sur_lot = jax.vmap(machine_scalaire, in_axes=(None, 0))
sorties = machine_sur_lot({"a": 2.0, "b": -1.0}, x_observe)
print(sorties)

[-0.99505475 -0.9866143  -0.96402758 -0.90514825 -0.76159416 -0.46211716
  0.          0.46211716  0.76159416]


## 6. Compilation avec `jit`

La transformation `jax.jit` demande à JAX de compiler le calcul décrit par une fonction pour des formes et des types donnés. La compilation a lieu lors du premier appel ; les appels suivants réutilisent le calcul compilé et peuvent être beaucoup plus rapides. Il vaut mieux vérifier d'abord la fonction sans compilation, puis lui appliquer `jit`.

La notation `@jax.jit`, placée juste avant la définition d'une fonction, est un **décorateur** (`'decorator'`) Python. Elle est équivalente à `f = jax.jit(f)`.

In [8]:
def pas_gradient_non_compile(parametres, x, z, alpha):
    valeur, gradient = jax.value_and_grad(cout_quadratique)(parametres, x, z)
    nouveaux_parametres = jax.tree_util.tree_map(
        lambda p, g: p - alpha * g, parametres, gradient
    )
    return nouveaux_parametres, valeur

pas_gradient = jax.jit(pas_gradient_non_compile)
alpha = 0.1
parametres_1, valeur_0 = pas_gradient(
    parametres, x_observe, z_observe, alpha
)
print(parametres_1, valeur_0)

{'a': Array(0.08333333, dtype=float64, weak_type=True), 'b': Array(-0.1, dtype=float64, weak_type=True)} 1.3333333333333333


## 7. Nombres aléatoires

JAX représente explicitement l'état d'un générateur aléatoire par une clé. Une clé ne doit pas être réutilisée pour deux tirages : on la partage avec `jax.random.split`. Cette convention rend l'aléatoire compatible avec les fonctions pures.

In [9]:
cle = jax.random.key(2026)
cle_x, cle_bruit = jax.random.split(cle)

x_apprentissage = jax.random.uniform(
    cle_x, shape=(64,), minval=-1.0, maxval=1.0
)
bruit = 0.05 * jax.random.normal(cle_bruit, shape=x_apprentissage.shape)
z_apprentissage = 2.0 * x_apprentissage - 1.0 + bruit

print(x_apprentissage[:5])
print(z_apprentissage[:5])

[-0.37989281 -0.9731084  -0.96121877  0.32897809  0.42529388]
[-1.76017051 -3.00735931 -2.95702727 -0.38710739 -0.16648465]


### Exercice 4

**Identification d'une machine affine.**

En partant de `parametres = {"a": 0.0, "b": 0.0}`, appliquez cent fois `pas_gradient` aux observations aléatoires ci-dessus.

1. Suivez la valeur du coût tous les dix pas.
2. Comparez les paramètres obtenus aux valeurs exactes $a=2$ et $b=-1$.
3. Recommencez avec plusieurs amplitudes du bruit.
4. Que se passe-t-il si la cible devient $z=x^2-1$ ? Est-ce un défaut de l'algorithme ou de la classe de machines choisie ?

In [ ]:
# À compléter.

## Bilan

Une transformation JAX reçoit une fonction et produit une nouvelle fonction : `grad` différentie, `vmap` vectorise et `jit` compile. Leur combinaison permettra ensuite de définir et d'ajuster des machines plus complexes sans écrire à la main leurs dérivées.